<h1 align="center"> Aplicações em Processamento de Linguagem Natural </h1>
<h2 align="center"> Aula 03 - Técnicas de Pré-Processamento de Texto e Similaridade</h2>
<h3 align="center"> Prof. Fernando Vieira da Silva MSc.</h3>

<h2> Técnicas para Pré-Processamento - Parte 2</h2>

<p>Uma vez que o texto já foi devidamente tratado, removendo stopwords e pontuações, e aplicando stemming ou lemmatization, agora precisamos contar a frequência das palavras (ou n-grams) que utilizaremos em seguida como atributos para as técnicas de aprendizado de máquina.</p>

<b>1. TF-IDF (Term Frequency - Inverse Document Frequency)</b>

<p><b>Term Frequency:</b> um termo que aparece muito em um documento, tende a ser um termo importante. Em resumo, divide-se o número de vezes em que um termo apareceu pelo maior número de vezes em que algum outro termo apareceu no documento.</p>

tf<sub>wd</sub> = f<sub>wd</sub> / m<sub>wd</sub>

onde:<br>
f<sub>wd</sub> é o número de vezes em que o termo <i>w</i> aparece no documento <i>d</i>.<br>
m<sub>wd</sub> é o maior valor de f<sub>wd</sub> obtido para algum termo do documento <i>d</i><br>

<p><b>Inverse Document Frequency:</b> um termo que aparece em poucos documentos pode ser um bom descriminante. Obtem-se dividindo o número de documentos pelo número de documentos em que o termo aparece.</p>

idf<sub>w</sub> = log<sub>2</sub>(n / n<sub>w</sub>)

onde:<br>
n é o número de documentos no corpus
n<sub>w</sub> é o número de documentos em que o termo <i>w</i> aparece.

Na prática, usa-se:

tf-idf = tf<sub>wd</sub> * idf<sub>w</sub>

Podemos calcular o TF-IDF de um corpus usando o pacote <b>scikit-learn</b>. Primeiramente, vamos abrir novamente o texto de Hamlet e armazenar as sentenças em uma ndarray do numpy (como se cada sentença fosse um documento do corpus):

In [ ]:
import nltk
import numpy as np
from nltk.tokenize import sent_tokenize

hamlet_raw = nltk.corpus.gutenberg.raw('shakespeare-hamlet.txt')

sents = sent_tokenize(hamlet_raw)

hamlet_np = np.array(sents)

print(hamlet_np.shape)


In [ ]:
hamlet_np

<p>Agora vamos definir uma função para tokenização pelo scikit-learn.</p>

In [ ]:
from nltk import pos_tag
from nltk.corpus import stopwords
from nltk.stem import WordNetLemmatizer
from nltk.tokenize import word_tokenize
import string
from nltk.corpus import wordnet

stopwords_list = stopwords.words('english')

lemmatizer = WordNetLemmatizer()

def my_tokenizer(doc):
    words = word_tokenize(doc)
    
    pos_tags = pos_tag(words)
    
    non_stopwords = [w for w in pos_tags if not w[0].lower() in stopwords_list]
    
    non_punctuation = [w for w in non_stopwords if not w[0] in string.punctuation]
    
    lemmas = []
    for w in non_punctuation:
        if w[1].startswith('J'):
            pos = wordnet.ADJ
        elif w[1].startswith('V'):
            pos = wordnet.VERB
        elif w[1].startswith('N'):
            pos = wordnet.NOUN
        elif w[1].startswith('R'):
            pos = wordnet.ADV
        else:
            pos = wordnet.NOUN
        
        lemmas.append(lemmatizer.lemmatize(w[0], pos))

    return lemmas
    
    

E essa função será chamada pelo objeto TfidfVectorizer

In [ ]:
from sklearn.feature_extraction.text import TfidfVectorizer

hamlet_raw = nltk.corpus.gutenberg.raw('shakespeare-hamlet.txt')

sents = sent_tokenize(hamlet_raw)

hamlet_np = np.array(sents)

tfidf_vectorizer = TfidfVectorizer(tokenizer=my_tokenizer)

tfs = tfidf_vectorizer.fit_transform(hamlet_np)

print(tfs.shape)

In [ ]:
print([k for k in tfidf_vectorizer.vocabulary_.keys()][:20])

In [ ]:
print(tfs[:50,:50])#linha 0 coluna 17

<b>2. TF-IDF de N-gramas</b>

Opcionalmente, podemos obter os atributos tf-idf de n-grams, combinando as classes CountVectorizer e TfidfTransformer. Em nosso exemplo, vamos utilizar apenas trigramas:

In [ ]:
from sklearn.feature_extraction.text import CountVectorizer
from sklearn.feature_extraction.text import TfidfTransformer

count_vect = CountVectorizer(ngram_range=(3,3))

n_gram_counts = count_vect.fit_transform(hamlet_np)

tfidf_transformer = TfidfTransformer()

tfs_ngrams = tfidf_transformer.fit_transform(n_gram_counts)

print(tfs_ngrams.shape)

<b>3. Redução de Dimensionalidade</b>

<p>A transformação do corpus em atributos contendo as frequências TF-IDF em geral resultará numa ndarray bastante esparsa, ou seja, com muitas dimensões. Porém, além de isso tornar o treinamento de algoritmos mais demorado e custoso (computacionalmente falando), muitas dessas dimensões provavelmente são pouco representativas ou mesmo podem causar ruído durante o treinamento. Para resolver esse problema, podemos aplicar uma técnica de redução de dimensionalidade simples chamada <b>Singular Value Decomposition (SVD)</b>. 

<p>Essa técnica transformará os vetores da matriz original, rotacionando e escalando-os, resultando em novas representações. A redução de dimensionalidade é feita ao manter apenas as <i>k</i> dimensões mais representativas que escolhermos. Outra vantagem dessa técnica é que as dimensões originais são, de certa forma, "combinadas", o que resulta em uma nova forma de representar a combinação de termos. No contexto de PLN, essa técnica é conhecida como <b>Latent Semantic Analysis (LSA)</b></p>

In [ ]:
from sklearn.decomposition import TruncatedSVD

svd_transformer = TruncatedSVD(n_components=1000)

svd_transformer.fit(tfs)

print(sorted(svd_transformer.explained_variance_ratio_)[::-1][:30])#lista de variância dos componentes, os 30 primeiros

<p>Agora vamos manter as dimensões até que a variância acumulada seja maior ou igual a 0.50.</p>

In [ ]:
cummulative_variance = 0.0
k = 0
for var in sorted(svd_transformer.explained_variance_ratio_)[::-1]:
    cummulative_variance += var
    if cummulative_variance >= 0.5:
        break
    else:
        k += 1
        
print(k)#143 colunas

<p>Transformarmos novamente, mas desta vez com o número de k componentes que obtemos anteriormente.</p>

In [ ]:
svd_transformer = TruncatedSVD(n_components=k)
svd_data = svd_transformer.fit_transform(tfs)
print(sorted(svd_transformer.explained_variance_ratio_)[::-1])

In [ ]:
print(svd_data.shape)

In [ ]:
print(svd_data)

<h2> Similaridade de Cosseno</h2>

<p>Uma vez que dois documentos são representados como vetores numéricos, é possível comparar a similaridade entre os documentos ao calcular o cosseno do ângulo entre esses documentos - uma medida de distância, não de amplitude. Para isso, basta resolver a equação do produto escalar entre os vetores, para encontrar o cosseno.</p>

<img src="http://s0.wp.com/latex.php?latex=++%5Cdisplaystyle++%5Cvec%7Ba%7D+%5Ccdot+%5Cvec%7Bb%7D+%3D+%5C%7C%5Cvec%7Ba%7D%5C%7C%5C%7C%5Cvec%7Bb%7D%5C%7C%5Ccos%7B%5Ctheta%7D+%5C%5C+%5C%5C++%5Ccos%7B%5Ctheta%7D+%3D+%5Cfrac%7B%5Cvec%7Ba%7D+%5Ccdot+%5Cvec%7Bb%7D%7D%7B%5C%7C%5Cvec%7Ba%7D%5C%7C%5C%7C%5Cvec%7Bb%7D%5C%7C%7D++&amp;bg=ffffff&amp;fg=000000&amp;s=0" alt="  \displaystyle  \vec{a} \cdot \vec{b} = \|\vec{a}\|\|\vec{b}\|\cos{\theta} \\ \\  \cos{\theta} = \frac{\vec{a} \cdot \vec{b}}{\|\vec{a}\|\|\vec{b}\|}  " title="  \displaystyle  \vec{a} \cdot \vec{b} = \|\vec{a}\|\|\vec{b}\|\cos{\theta} \\ \\  \cos{\theta} = \frac{\vec{a} \cdot \vec{b}}{\|\vec{a}\|\|\vec{b}\|}  " class="latex">

<img class=" wp-image-2582 " title="vector_space" src="http://blog.christianperone.com/wp-content/uploads/2013/09/vector_space.png" alt="" width="504" height="378">
<p fontsize=8>Fonte: http://blog.christianperone.com</p>

In [11]:
    documents = (
    "The sky is blue",
    "The sun is bright",
    "The sun in the sky is bright",
    "We can see the shining sun, the bright sun"
    )

In [12]:
from sklearn.feature_extraction.text import TfidfVectorizer
tfidf_vectorizer = TfidfVectorizer()
tfidf_matrix = tfidf_vectorizer.fit_transform(documents)
print(tfidf_matrix.shape)

(4, 11)


In [13]:
from sklearn.metrics.pairwise import cosine_similarity

query_vect = tfidf_vectorizer.transform(["The sun is red in the sky"])#ele segmenta e tokeniza automático, mas é bom criar o próprio

cosine_similarity(query_vect, tfidf_matrix)#próx de um é mais similar, é a terceira frase

array([[0.55245238, 0.59222761, 0.9467919 , 0.37986898]])

In [ ]:
import numpy as np
desired_width = 320
pd.set_option('display.width', desired_width)
np.set_printoptions(linewidth=desired_width)
pd.set_option('display.max_columns',20)
pd.set_option('display.max_rows', None)

<p><b>Exercício 3:</b>Escreva um chatbot que, dado uma pergunta em Inglês, encontre uma pergunta mais parecida no corpus de perguntas e respostas disponível no Kaggle (https://www.kaggle.com/rtatman/questionanswer-dataset#S08_question_answer_pairs.txt) e exiba a resposta.

In [1]:
from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.metrics.pairwise import cosine_similarity
import pandas as pd
import numpy as np

In [2]:
df = pd.read_csv('../input/S08_question_answer_pairs.txt', sep='\t')
#df2 = df['ArticleTitle\tQuestion\tAnswer\tDifficultyFromQuestioner\tDifficultyFromAnswerer\tArticleFile']
df.head(100)

,ArticleTitle,Question,Answer,DifficultyFromQuestioner,DifficultyFromAnswerer,ArticleFile
0,Abraham_Lincoln,Was Abraham Lincoln the sixteenth President of...,yes,easy,easy,S08_set3_a4
1,Abraham_Lincoln,Was Abraham Lincoln the sixteenth President of...,Yes.,easy,easy,S08_set3_a4
2,Abraham_Lincoln,Did Lincoln sign the National Banking Act of 1...,yes,easy,medium,S08_set3_a4
3,Abraham_Lincoln,Did Lincoln sign the National Banking Act of 1...,Yes.,easy,easy,S08_set3_a4
4,Abraham_Lincoln,Did his mother die of pneumonia?,no,easy,medium,S08_set3_a4
5,Abraham_Lincoln,Did his mother die of pneumonia?,No.,easy,easy,S08_set3_a4
6,Abraham_Lincoln,How many long was Lincoln's formal education?,18 months,medium,easy,S08_set3_a4
7,Abraham_Lincoln,How many long was Lincoln's formal education?,18 months.,medium,medium,S08_set3_a4
8,Abraham_Lincoln,When did Lincoln begin his political career?,1832,medium,easy,S08_set3_a4
9,Abraham_Lincoln,When did Lincoln begin his political career?,1832.,medium,medium,S08_set3_a4


In [3]:
df.info()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 1715 entries, 0 to 1714
Data columns (total 6 columns):
ArticleTitle                1715 non-null object
Question                    1696 non-null object
Answer                      1475 non-null object
DifficultyFromQuestioner    1224 non-null object
DifficultyFromAnswerer      1473 non-null object
ArticleFile                 1713 non-null object
dtypes: object(6)
memory usage: 80.5+ KB


In [4]:
print('Antes:', df.shape)
df.drop_duplicates()
print('Depois:', df.shape)

Antes: (1715, 6)
Depois: (1715, 6)


In [5]:
df.isna().sum()

ArticleTitle                  0
Question                     19
Answer                      240
DifficultyFromQuestioner    491
DifficultyFromAnswerer      242
ArticleFile                   2
dtype: int64

In [6]:
df = df[['Question','Answer']]
df.head()

,Question,Answer
0,Was Abraham Lincoln the sixteenth President of...,yes
1,Was Abraham Lincoln the sixteenth President of...,Yes.
2,Did Lincoln sign the National Banking Act of 1...,yes
3,Did Lincoln sign the National Banking Act of 1...,Yes.
4,Did his mother die of pneumonia?,no


In [7]:
df.dropna(inplace=True)
df.isna().sum()

Question    0
Answer      0
dtype: int64

In [56]:
q = df.Question.tolist()
a = df.Answer.tolist()
q

['Was Abraham Lincoln the sixteenth President of the United States?',
 'Was Abraham Lincoln the sixteenth President of the United States?',
 'Did Lincoln sign the National Banking Act of 1863?',
 'Did Lincoln sign the National Banking Act of 1863?',
 'Did his mother die of pneumonia?',
 'Did his mother die of pneumonia?',
 "How many long was Lincoln's formal education?",
 "How many long was Lincoln's formal education?",
 'When did Lincoln begin his political career?',
 'When did Lincoln begin his political career?',
 'What did The Legal Tender Act of 1862 establish?',
 'What did The Legal Tender Act of 1862 establish?',
 'Who suggested Lincoln grow a beard?',
 'Who suggested Lincoln grow a beard?',
 'When did the Gettysburg address argue that America was born?',
 'When did the Gettysburg address argue that America was born?',
 'Did Lincoln beat John C. Breckinridge in the 1860 election?',
 'Did Lincoln beat John C. Breckinridge in the 1860 election?',
 'Was Abraham Lincoln the first Pr

In [9]:
from nltk import pos_tag
from nltk.corpus import stopwords
from nltk.stem import WordNetLemmatizer
from nltk.tokenize import word_tokenize
import string
from nltk.corpus import wordnet

stopwords_list = stopwords.words('english')

lemmatizer = WordNetLemmatizer()

def my_tokenizer(doc):
    words = word_tokenize(doc)
    
    pos_tags = pos_tag(words)
    
    non_stopwords = [w for w in pos_tags if not w[0].lower() in stopwords_list]
    
    non_punctuation = [w for w in non_stopwords if not w[0] in string.punctuation]
    
    lemmas = []
    for w in non_punctuation:
        if w[1].startswith('J'):
            pos = wordnet.ADJ
        elif w[1].startswith('V'):
            pos = wordnet.VERB
        elif w[1].startswith('N'):
            pos = wordnet.NOUN
        elif w[1].startswith('R'):
            pos = wordnet.ADV
        else:
            pos = wordnet.NOUN
        
        lemmas.append(lemmatizer.lemmatize(w[0], pos))

    return lemmas

In [57]:
#tok = sent_tokenize(q)
qn = np.array(q)
qn[0]

'Was Abraham Lincoln the sixteenth President of the United States?'

In [44]:
#tok = sent_tokenize(q)
tfidf_vectorizer_q = TfidfVectorizer(tokenizer=my_tokenizer)
tfidf_matrix_q = tfidf_vectorizer.fit_transform(q)
print(tfidf_matrix_q.shape)

(1475, 1958)


In [45]:
tfidf_vectorizer_a = TfidfVectorizer(tokenizer=my_tokenizer)
tfidf_matrix_a = tfidf_vectorizer.fit_transform(a)
print(tfidf_matrix_a.shape)

(1475, 1484)


In [67]:
from sklearn.decomposition import TruncatedSVD

svd_transformer = TruncatedSVD(n_components=1000)

svd_transformer.fit(tfidf_matrix_a)

print(sorted(svd_transformer.explained_variance_ratio_)[::-1][:30])#lista de variância dos componentes, os 30 primeiros

[0.2789655209987825, 0.005432411068845237, 0.004468005849943647, 0.00408689543602438, 0.004083392072237264, 0.004082093797454486, 0.004075645906725975, 0.003923800562888356, 0.0038808261640369108, 0.003867616941249538, 0.0038292328046277183, 0.003743845572787127, 0.0036380786151258494, 0.0036207853896607647, 0.0034697598339999518, 0.003424992898232647, 0.003418842680841326, 0.003410805730315657, 0.0034090085700726144, 0.00338617890677555, 0.003375660933166749, 0.0033409048855112815, 0.0032944255458493216, 0.0032291715275325093, 0.0031717360593734607, 0.003130990383760177, 0.003068376469042071, 0.003001713246613605, 0.0029882592712896514, 0.002950768705626418]


In [70]:
print(svd_transformer.shape)

AttributeError: 'TruncatedSVD' object has no attribute 'shape'

In [37]:
print([k for k in tfidf_vectorizer.vocabulary_.keys()][:100])

['abraham', 'lincoln', 'sixteenth', 'president', 'united', 'state', 'sign', 'national', 'banking', 'act', '1863', 'mother', 'die', 'pneumonia', 'many', 'long', "'s", 'formal', 'education', 'begin', 'political', 'career', 'legal', 'tender', '1862', 'establish', 'suggest', 'grow', 'beard', 'gettysburg', 'address', 'argue', 'america', 'bear', 'beat', 'john', 'c.', 'breckinridge', '1860', 'election', 'first', 'start', '1832', 'ever', 'represent', 'alton', 'sangamon', 'railroad', 'county', 'born', 'serve', 'assassinate', 'win', 'general', 'charge', 'battle', 'antietam', 'issue', 'emancipation', 'proclamation', 'scholar', 'rank', 'among', 'top', 'three', '18', 'month', 'school', 'chosen', 'presidential', 'candidate', 'old', '1816', 'photgraph', 'take', 'trail', 'use', 'farmer', 'almanac', 'live', 'frontier', 'wife', 'family', 'support', 'slavery', 'noted', 'contribution', 'theory', 'molarity', 'molecular', 'weight', 'graduate', 'ecclesiastical', 'law', 'early', 'age', '20', 'practice', 'publ

In [36]:
print(tfidf_matrix[:50,:50])#linha 0 coluna 17

  (2, 46)	0.47386733330505154
  (3, 46)	0.47386733330505154
  (6, 4)	0.24857998835147035
  (7, 4)	0.24857998835147035
  (10, 45)	0.48160953324253425
  (11, 45)	0.48160953324253425
  (16, 44)	0.3962093491634134
  (17, 44)	0.3962093491634134
  (20, 40)	0.5106893054831116
  (21, 40)	0.5106893054831116
  (30, 44)	0.5747922418515101
  (31, 44)	0.5747922418515101
  (37, 30)	0.6013209916013743
  (38, 44)	0.44959497433572787
  (39, 35)	0.708799967800982
  (41, 4)	0.26813377615268247
  (44, 4)	0.43741259135279054


In [66]:
#tfidf_vectorizer_q = TfidfVectorizer(tokenizer=my_tokenizer)
tfidf_matrix_q = tfidf_vectorizer.fit_transform(['Was Abraham Lincoln the sixteenth President of the United States?'])
cosine_similarity(tfidf_matrix_q, tfidf_matrix_a)

ValueError: Incompatible dimension for X and Y matrices: X.shape[1] == 6 while Y.shape[1] == 1484

In [15]:
tfidf_matrix

<4x11 sparse matrix of type '<class 'numpy.float64'>'
	with 21 stored elements in Compressed Sparse Row format>

In [21]:
import nltk
from sklearn.feature_extraction.text import TfidfVectorizer
#from nltk.tokenize import sent_tokenize

hamlet_raw = nltk.corpus.gutenberg.raw('shakespeare-hamlet.txt')

sents = sent_tokenize(hamlet_raw)

hamlet_np = np.array(sents)

tfidf_vectorizer = TfidfVectorizer(tokenizer=my_tokenizer)

tfs = tfidf_vectorizer.fit_transform(hamlet_np)

print(tfs.shape)

(2355, 4305)


In [24]:
hamlet_raw

"[The Tragedie of Hamlet by William Shakespeare 1599]\n\n\nActus Primus. Scoena Prima.\n\nEnter Barnardo and Francisco two Centinels.\n\n  Barnardo. Who's there?\n  Fran. Nay answer me: Stand & vnfold\nyour selfe\n\n   Bar. Long liue the King\n\n   Fran. Barnardo?\n  Bar. He\n\n   Fran. You come most carefully vpon your houre\n\n   Bar. 'Tis now strook twelue, get thee to bed Francisco\n\n   Fran. For this releefe much thankes: 'Tis bitter cold,\nAnd I am sicke at heart\n\n   Barn. Haue you had quiet Guard?\n  Fran. Not a Mouse stirring\n\n   Barn. Well, goodnight. If you do meet Horatio and\nMarcellus, the Riuals of my Watch, bid them make hast.\nEnter Horatio and Marcellus.\n\n  Fran. I thinke I heare them. Stand: who's there?\n  Hor. Friends to this ground\n\n   Mar. And Leige-men to the Dane\n\n   Fran. Giue you good night\n\n   Mar. O farwel honest Soldier, who hath relieu'd you?\n  Fra. Barnardo ha's my place: giue you goodnight.\n\nExit Fran.\n\n  Mar. Holla Barnardo\n\n   Bar. 

In [25]:
sents

['[The Tragedie of Hamlet by William Shakespeare 1599]\n\n\nActus Primus.',
 'Scoena Prima.',
 'Enter Barnardo and Francisco two Centinels.',
 'Barnardo.',
 "Who's there?",
 'Fran.',
 'Nay answer me: Stand & vnfold\nyour selfe\n\n   Bar.',
 'Long liue the King\n\n   Fran.',
 'Barnardo?',
 'Bar.',
 'He\n\n   Fran.',
 'You come most carefully vpon your houre\n\n   Bar.',
 "'Tis now strook twelue, get thee to bed Francisco\n\n   Fran.",
 "For this releefe much thankes: 'Tis bitter cold,\nAnd I am sicke at heart\n\n   Barn.",
 'Haue you had quiet Guard?',
 'Fran.',
 'Not a Mouse stirring\n\n   Barn.',
 'Well, goodnight.',
 'If you do meet Horatio and\nMarcellus, the Riuals of my Watch, bid them make hast.',
 'Enter Horatio and Marcellus.',
 'Fran.',
 'I thinke I heare them.',
 "Stand: who's there?",
 'Hor.',
 'Friends to this ground\n\n   Mar.',
 'And Leige-men to the Dane\n\n   Fran.',
 'Giue you good night\n\n   Mar.',
 "O farwel honest Soldier, who hath relieu'd you?",
 'Fra.',
 "Barnar

In [28]:
tfidf_vectorizer

TfidfVectorizer(analyzer='word', binary=False, decode_error='strict',
        dtype=<class 'numpy.float64'>, encoding='utf-8', input='content',
        lowercase=True, max_df=1.0, max_features=None, min_df=1,
        ngram_range=(1, 1), norm='l2', preprocessor=None, smooth_idf=True,
        stop_words=None, strip_accents=None, sublinear_tf=False,
        token_pattern='(?u)\\b\\w\\w+\\b',
        tokenizer=<function my_tokenizer at 0x7ffb68e3b9d8>, use_idf=True,
        vocabulary=None)

In [29]:
print([k for k in tfidf_vectorizer.vocabulary_.keys()][:20])

['tragedie', 'hamlet', 'william', 'shakespeare', '1599', 'actus', 'primus', 'scoena', 'prima', 'enter', 'barnardo', 'francisco', 'two', 'centinels', "'s", 'fran', 'nay', 'answer', 'stand', 'vnfold']


In [30]:
print(tfs[:50,:50])#linha 0 coluna 17

  (0, 17)	0.4098405429500896
  (4, 5)	1.0
  (12, 6)	0.25569565102708125
  (13, 6)	0.24183188573710981
  (22, 5)	0.5865767987601467
  (27, 0)	0.226773205328385
  (29, 5)	0.28458175550618814
  (37, 5)	0.26402019659267234
  (37, 0)	0.23602778459784524
  (39, 6)	0.13875455587670743
  (40, 13)	0.35297267044370917
  (41, 25)	0.31475626155215114
  (43, 5)	0.14416024946136125
  (46, 5)	0.32299630350190056
